# 05. 점수표 나눠 돌리기 — 실행 전·합치기 전 점검

대상: 팀원 4명 (11/25 회의에서 같이 보고, 각자 자기 몫을 돌리기 전에 실행)

11/24 팀장 결정 D6(회의기록.md): 신호 18개 단독 점수표를 **4명이 나눠 동시에** 돌린다. 나누는 것은 돌리는 일뿐이고, 엔진·규칙·명령은 모두 같다.
이 노트북은 그 약속이 지켜지는지 **직접 돌려서** 확인한다. 결과 숫자(점수·Edge)는 여기서 보지 않는다 — 그것은 합친 뒤 `05_신호점수표.ipynb` 에서 본다.

1. 새 계산 부품 테스트 (대조군·t값·카이제곱·고르기 규칙·합치기 점검)
2. 내 컴퓨터 코드·데이터가 엔진 v2 고정값과 같은가 (11/24 '시가 0 매수' 결함을 고친 판. v1 기준 파일은 지우지 않고 보관) (다르면 실행이 거부된다)
3. 4명 나눈 표
4. 결과 묶음 점검 (`results/scorecard_parts/`, 18개가 다 모였을 때 합치기 전에)
5. 시험 실행 기록 (팀장, 11/24)

이 노트북은 데이터 파일을 저장하지 않는다 (§9). 결과 파일은 명령(`python -m indexbeater optimize`)만 만든다.

In [1]:
# 이 칸을 맨 먼저 실행한다.
# 노트북이 어느 폴더에서 열려도, indexbeater 폴더가 있는 프로젝트 폴더를 찾아서 길 목록(sys.path)에 넣는다.
import os
import sys

root = os.getcwd()
while not os.path.isdir(os.path.join(root, "indexbeater")) and os.path.dirname(root) != root:
    root = os.path.dirname(root)          # 못 찾으면 한 칸 위 폴더로 올라가서 다시 찾는다
sys.path.insert(0, root)
os.chdir(root)
print("프로젝트 폴더:", root)

프로젝트 폴더: C:\Users\gram\Desktop\project\buysignal


## 1. 새 계산 부품 테스트

- `tests/test_metrics.py`: 대조군에서 신호 종목 빼기(중복_무시 포함), 대조 종목 없을 때 NaN, 상장폐지 최악(−100%), 5분위, 20일 블록 t값, 카이제곱, 승률·손익비, 상위 5종목, 체결 불가 비율, 최대 낙폭 — 손 계산과 비교
- `tests/test_optimize.py`: 이웃(한 축 한 칸), 중앙값으로 고르기, 300건 미만 제외, 동점 → 기본값 가까운 쪽, 전부 부족 → 기본값, 지문 비교, 나눈 표가 18개를 한 번씩 덮는지, 합치기 거부 조건

맨 아래 줄이 `passed` 이고 `failed` 가 없으면 통과.

In [2]:
import subprocess

r = subprocess.run([sys.executable, "-m", "pytest", "tests/test_metrics.py", "tests/test_optimize.py", "--color=no"],
                   capture_output=True, text=True, encoding="utf-8", errors="replace")
print(r.stdout[-1500:] or r.stderr[-1500:])
assert r.returncode == 0, "테스트 실패 — 팀장에게 알려 주세요"


................................                                         [100%]
32 passed in 2.17s



## 2. 내 컴퓨터가 엔진 v2 와 같은가

`results/engine_v2_reference.csv` 의 지문 (노트북 04 를 끝까지 돌리면 생긴다)(엔진 6개·데이터)과 지금 내 파일의 지문을 비교한다. 하나라도 다르면 `optimize` 명령이 실행을 거부한다.
새로 만든 `metrics.py`·`scorecard.py`·`scoring.py` 지문도 같이 보인다 — 4명 숫자가 같아야 한다 (합치기에서 확인).

In [3]:
import pandas as pd

from indexbeater import optimize as O
from indexbeater import scorecard as SC

now = SC.fingerprints(with_data=True)
ref = O.reference_fingerprints()
tbl = pd.DataFrame({"지금": now, f"엔진 {O.ENGINE_VERSION} 고정값": pd.Series(ref)})
tbl["같음"] = [(k not in ref) or (now.get(k) == ref.get(k)) for k in tbl.index]
display(tbl)
bad = O.check_frozen(now, ref)
print(f"엔진 {O.ENGINE_VERSION} 기준과 같음 ✓" if not bad else "다름 ✗ → " + "; ".join(bad))
print("git:", O.git_commit())

,지금,엔진 v2 고정값,같음
data/krx_adjusted.parquet,f73644072205,f73644072205,True
indexbeater/config.py,4fda0ea8b5a8,4fda0ea8b5a8,True
indexbeater/data_io.py,6c8f4c9b729e,6c8f4c9b729e,True
indexbeater/engine.py,c1feda245a03,c1feda245a03,True
indexbeater/exits.py,bbcbb26f4ab3,bbcbb26f4ab3,True
indexbeater/indicators.py,9a4a81a80982,9a4a81a80982,True
indexbeater/metrics.py,256b7c5c1a7d,NaN,True
indexbeater/scorecard.py,0a15bd9696e6,NaN,True
indexbeater/scoring.py,be21a465597f,NaN,True
indexbeater/signals.py,78db6954cfdf,78db6954cfdf,True


엔진 v2 기준과 같음 ✓
git: fcf6366 (고친 파일 있음)


## 3. 4명 나눈 표

**일의 양**(세트 수 × H 3개, 신호가 켜지는 수)으로 맞췄다. 결과와는 상관없이 정했다.
내 몫 돌리기: `python -m indexbeater optimize --part 내번호` (프로젝트 폴더에서, 터미널로)

In [4]:
O.main(["--list"])

  1. 송재영: 15.vol_breakout, 18.candle, 17.obv, 13.squeeze, 6.sar  (세트 14개)
  2. 김동겸: 12.bollinger, 3.macd, 2.alignment, 14.donchian, 4.ichimoku  (세트 19개)
  3. 김승찬: 7.rsi, 16.volume_breakout, 5.adx_di, 8.stochastic  (세트 20개)
  4. 김정철: 9.disparity, 1.ma_cross, 10.cci, 11.williams_r  (세트 19개)


0

## 4. 결과 묶음 점검 (합치기 전에)

4명의 결과 묶음을 `results/scorecard_parts/` 에 모은 뒤 이 칸을 실행한다. 문제가 없으면 `python -m indexbeater optimize --merge`.
- 18개 신호가 다 있는가, 파일 7개가 다 있는가
- 모든 묶음의 지문(코드 9개·데이터)이 서로 같은가, 지금 내 파일과 같은가, 엔진 v2 와 같은가
- 포트폴리오를 건너뛴 묶음은 없는가

In [5]:
from pathlib import Path

part_dir = SC.PART_DIR
if not part_dir.exists():
    print("아직 결과 묶음이 없습니다:", part_dir)
else:
    problems, metas = O.check_parts(part_dir, now=now)
    rows = [{"신호": n, "돌린사람": m.get("돌린사람"), "git": m.get("git"), "돌린시각": m.get("돌린시각"),
             "걸린초": m.get("걸린초")} for n, m in metas.items()]
    if rows:
        display(pd.DataFrame(rows))
    print("합쳐도 됨 ✓" if not problems else "아직 합치면 안 됨 ✗\n - " + "\n - ".join(problems))

아직 결과 묶음이 없습니다: C:\Users\gram\Desktop\project\buysignal\results\scorecard_parts


## 4-2. 똑같이 나오는가 (재현 확인)

같은 엔진·같은 데이터·같은 프로그램 판이면 결과는 **소수점까지 똑같아야** 한다 (엔진에 무작위가 없고, 같은 값끼리 순서는 종목코드로 정한다 — 회의기록 11/24 재현 규칙).
한 명이 `python -m indexbeater optimize --all --out results/scorecard_all` 로 18개를 한 번에 돌린 뒤 이 칸을 실행해, 4명이 나눠 돌린 묶음과 비교한다.
- 같음 = 통과 / 1e-9 이하 차이 = 프로그램 판 차이 (판을 기록) / 그보다 크거나 줄 수·빈칸 위치가 다름 = **코드 결함** → 결과를 쓰지 않는다

In [6]:
all_dir = Path("results/scorecard_all")
if part_dir.exists() and all_dir.exists():
    O.compare(part_dir, all_dir)
else:
    print("비교할 폴더가 아직 없습니다:", part_dir if not part_dir.exists() else all_dir)

비교할 폴더가 아직 없습니다: C:\Users\gram\Desktop\project\buysignal\results\scorecard_parts


## 5. 시험 실행 기록 (팀장, 2026-11-24)

명령이 끝까지 도는지, 파일이 다 생기는지만 확인했다. **결과 숫자는 보지 않았다** (결과를 보고 규칙을 바꾸지 않기 위해).

- 돌린 신호: ichimoku (S04) 1개, 결과는 임시 폴더(scratchpad/parts_smoke)에 저장 — `results/` 에는 아무것도 만들지 않음
- 걸린 시간: 약 103초 (대조군 바탕 약 60초 + 신호 약 32초 + 포트폴리오 6번)
- 파일 7개 모두 생김. 줄 수: 그리드 6 · 학습구간 36 · 고른 세트 36 · 점수표 12 · 검증 매매 55,396 · 포트폴리오 일별 7,362
- → 18개를 한 컴퓨터로 돌리면 약 20분, 4명이 나누면 1명당 약 5~7분 예상

**이상한 것 (실행 보류):** 자료에 `시가·고가·저가 = 0, 거래량 > 0` 인 칸이 53개(40종목) 있다. 엔진 v1 은 이날을 "살 수 있는 날"로 보고 **0원에 산다** → 수익이 무한대(inf)가 된다.
- 예: 003190 (2013-03-21) 등. 파는 쪽은 이미 막혀 있음(하한가처럼 처리).
- 그래서 점수표 코드에 **점검 장치**를 넣었다: 무한대 수익이 하나라도 있으면 숫자를 만들지 않고 멈춘다 (`BadTradeError`).
- 고치는 방법은 11/25 회의에서 4명이 정한다 (엔진 v2 로 바꾸면 `engine_v2_reference.csv` 새로 만들고 노트북 04 를 다시 돌린다).

**0원 매수가 신호마다 몇 건인가 (팀장, 11/24, 모든 세트 × H 5·10·20, 체결 매매만 셈):**

| 신호 | 0원 매수 | 신호 | 0원 매수 | 신호 | 0원 매수 |
|---|---|---|---|---|---|
| ma_cross | 3 | alignment | 0 | macd | 15 |
| ichimoku | 0 | adx_di | 0 | sar | 3 |
| rsi | 0 | stochastic | 3 | disparity | 9 |
| cci | 6 | williams_r | 0 | bollinger | 9 |
| squeeze | 0 | donchian | 0 | vol_breakout | 27 |
| volume_breakout | 0 | obv | 7 | candle | 0 |

- 합계 82건, 18개 중 9개 신호에 섞임 → 이대로는 9개 신호의 점수표가 멈춘다.
- 노트북 04 기준 매매(RSI 14·30, H=10)에는 0건.

**재검토 (팀장, 11/24 저녁) — 고친 것**
- 대조군 바탕(유동성 통과 전 종목)에도 0원 매수가 H마다 10건 있다 → 멈춤 장치 때문에 **지금은 18개 모두 3단계에서 멈춘다** (직접 돌려 확인: `--signals ichimoku` → "대조군 바탕 H=5: 무한대 10건", 종료코드 1, 결과 파일 안 만듦).
- `scorecard.py` 고른 세트의 학습거래수가 빈칸이면 멈추던 코드(`int(NaN or 0)`)를 고침.
- `optimize.py`: 신호 계산 중 멈춤도 깔끔히 알림, meta 에 프로그램 판(python·numpy·pandas·scipy·pyarrow) 기록, 합칠 때 판이 다르면 '주의' 표시, `--all`(18개 한 번에)·`--compare 폴더A 폴더B`(같은지 비교) 추가. 테스트 4개 추가 → 전체 126개 통과.

**재현 시험 (팀장, 11/24)** — 같은 엔진이면 똑같이 나오는가
- A: ichimoku 만 따로 / B: 다른 프로그램에서 cci 먼저 돌리고 ichimoku 를 두 번째로 (나눠 돌릴 때처럼 공통 준비를 이어 씀)
- 이 시험에서만 멈춤 장치를 껐다 (값은 쓰지 않고 같은지만 봄).
- 결과: 표 6개 모두 **"같음"** — 가장 큰 차이 0 (소수점 끝자리까지 같음). 걸린 시간 A 118초, B 174초.
- → 같은 컴퓨터에서는 돌리는 순서·묶음과 상관없이 똑같다. 다른 컴퓨터끼리는 4명이 돌린 뒤 `--compare` 로 확인한다.

**엔진 v2 로 고친 뒤 다시 확인 (팀장, 2026-11-24)** — 11/24 팀장 결정: 결함은 ① 엔진 v2, 기본값은 ② 교과서 표준값 (회의기록 '엔진 v2' 절)

- 고친 것: `engine.py` 만. 시가 0 인 날은 거래정지일처럼 사지도 팔지도 않는다 + 그날의 저가 0 으로 손절하지 않는다. 테스트 4개 추가 → 전체 130개 통과.
- 노트북 04: 2번 실행 모두 30/30 True. 새 기준 파일 `results/engine_v2_reference.csv` (engine.py `c1feda245a03`). v1 기준 파일은 그대로 보관.
- **18개 한 번에** (`--all`, 임시 폴더 scratchpad/all_v2): 끝까지 돎, 종료코드 0, 591초. 멈춤 장치(`BadTradeError`) 한 번도 안 걸림 → 0원 매수가 없어졌다.
  - 묶음 점검(`check_parts`): 18개 모두 있음, 파일 7개씩 다 있음, 지문 모두 같음 → 문제 없음.
- **나눠 돌리기 재현** (`--part 1` 을 따로, 임시 폴더 scratchpad/part1_v2, 246초): `--compare` 로 한 번에 돌린 것과 비교 → sar·squeeze·vol_breakout·obv·candle **5개 모두 같음** (vol_breakout 은 v1 에서 0원 매수가 가장 많던 신호).
- 이번에도 결과 숫자(점수·Edge)는 보지 않았다. `results/` 에는 v2 기준 파일 말고 아무것도 만들지 않았다.
- 남은 것: 4명이 각자 컴퓨터에서 노트북 04 → 이 노트북 → `--part 번호`. 다른 컴퓨터끼리 같은지는 합치기 전에 `--compare` 로 본다.
